## Publication and reproducibility notes

This notebook is part of the research source code for the **Comparative Disease Prediction** project.

- The original experiment logic and code-cell source are retained in this publication copy.
- Saved cell outputs have been cleared to remove machine-specific filesystem paths, verbose logs, and embedded data previews. This does **not** change the code or the experiment results recorded in the project documentation and reference-results files.
- The input datasets are not included in this repository. Obtain the permitted source files separately, review their licence/terms, and place them at the paths expected by the configuration cells.
- Run this notebook in a dedicated Python environment with the compatible dependencies installed. Do not assume the latest package versions are compatible with persisted model artefacts.
- Re-running the notebook performs the experiment (including CI testing and model fitting); it is not required to launch the separately packaged prediction application, which loads the saved trained models.
- This is a research prototype, not a clinical diagnostic system.

### Required input files

- `Train_Only_Diseases.csv`
- `Train_Only_Symptoms.csv`
- `Test_Only_Diseases.csv`
- `Test_Only_Symptoms.csv`

### Experiment summary

The notebook performs disease-specific CI screening, unions the retained symptoms into one deterministic feature matrix, and trains Logistic Regression, Random Forest, and XGBoost models. It also evaluates an all-symptoms baseline. The archived experiment used 4,920 training rows and 42 external test rows; this protocol should be described separately from the Stable-FCI split rather than treated as a directly matched test protocol.

### Data and results policy

Raw datasets, private files, environment folders, and machine-specific paths are intentionally not bundled. See the repository's data and reproducibility documentation for instructions on obtaining inputs and interpreting the archived results.


In [ ]:
!pip install -q xgboost


## STAR CI-Selected Symptom Screening and Multiclass Disease Classification

This notebook implements disease-specific conditional-independence screening, constructs a common CI-selected symptom set, and evaluates three downstream classifiers: Logistic Regression, Random Forest, and XGBoost. It also includes an all-symptoms baseline and evaluation diagnostics.


## 1. Imports

In [ ]:
import os
import io
import textwrap
import warnings
import contextlib
import multiprocessing as mp
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

import joblib
import numpy as np
import pandas as pd
from scipy.stats import chi2
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, log_loss, top_k_accuracy_score,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 2. Configuration


`ALPHA` controls the CI screening stage and is used as the significance
level for every G-square conditional-independence test.

`RANDOM_STATE` is kept as a reproducibility setting for the Logistic
Regression configuration. With the current `lbfgs` solver, the fit does not
depend on random initialization; the parameter is retained so that the
experiment configuration remains explicit and reproducible if the solver
or model settings are changed later. `RANDOM_STATE` also seeds the
cross-validation fold split described below, so the folds themselves are
reproducible.

`C_GRID` and `CV_FOLDS` control the overfitting-prevention step added in
Section 9: instead of fitting Logistic Regression at a single, arbitrarily
chosen regularization strength, `C` (inverse L2 regularization strength --
smaller values mean *stronger* regularization, i.e. a simpler, less
overfit-prone model) is selected by stratified cross-validation **within
the training data only**. This does not touch the test set and does not
change the model family (still multinomial `LogisticRegression`, still the
`lbfgs` solver) -- it only picks a better-justified value for one
hyperparameter instead of leaving it at scikit-learn's default.


In [ ]:
ALPHA = 0.05          # significance level for every G-square test (CI stage)
RANDOM_STATE = 42     # explicit experiment setting; retained for reproducibility

# --- Overfitting-prevention: cross-validated regularization strength ---
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]  # candidate inverse L2 regularization strengths
#           ^^^^^ added for stronger-regularization overfitting control (purely additive vs. the original grid)
CV_FOLDS = 5                             # stratified CV folds, searched on TRAINING data only

# --- Degenerate-test policy (unchanged from the reference CI notebook) ---
# VALID      : normal test, all conditioning strata usable.
# DEGENERATE : some stratum was structurally degenerate and excluded, but at
#              least one usable stratum remained -> a p-value IS computed.
# INVALID    : no usable stratum/data at all -> no p-value can be computed;
#              the decision is fixed to "DEPENDENT" so an unevaluable test can
#              never remove an edge, and Stage 2 continues to the next
#              paired variable.


## 3. Input file paths

In [ ]:
TRAIN_DISEASE_PATH  = "Train_Only_Diseases.csv"
TRAIN_SYMPTOMS_PATH  = "Train_Only_Symptoms.csv"
TEST_DISEASE_PATH    = "Test_Only_Diseases.csv"
TEST_SYMPTOMS_PATH   = "Test_Only_Symptoms.csv"

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join("results", f"run_{RUN_TIMESTAMP}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory for this run: {OUTPUT_DIR}")


## 4. Load datasets

In [ ]:
def load_csv(path: str, label: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} not found at: {path}")
    df = pd.read_csv(path)
    print(f"Loaded {label}: {df.shape[0]} rows x {df.shape[1]} columns")
    return df


train_disease_raw  = load_csv(TRAIN_DISEASE_PATH,  "training disease dataset")
train_symptoms     = load_csv(TRAIN_SYMPTOMS_PATH, "training symptoms dataset")
test_disease_raw   = load_csv(TEST_DISEASE_PATH,   "test disease dataset")
test_symptoms      = load_csv(TEST_SYMPTOMS_PATH,  "test symptoms dataset")


## 5. Validate datasets

Checks, in order:

1. **Binary + no missing values** in every disease/symptom column, train and
   test alike.
2. **Row alignment**: disease/symptom row counts match within train, and
   within test. This confirms the row *counts* line up; it does not by
   itself prove row 0 of the disease file corresponds to row 0 of the
   symptoms file, and so on. Neither of these CSVs carries a patient-ID
   column, so that correspondence cannot be independently verified from the
   files alone — it is a **documented assumption**: these files are assumed
   to have been produced by splitting a single source dataframe into
   disease-only and symptom-only columns without any reordering or
   shuffling in between. If that assumption does not hold for how your
   files were generated, the row-count check here will not catch it.
3. **Exactly one disease per patient** (one-hot) in both train and test —
   zero or multiple diseases in a row is a hard error, never silently fixed.
4. **Disease-class correspondence** between train and test. The two files
   here use different capitalisation/whitespace for the same 41 diseases
   (e.g. `disease_Bronchial Asthma` vs `disease_bronchial asthma`) — this is
   detected explicitly by normalizing (strip + lowercase) for *comparison
   only*, every test column is matched to exactly one train column, the
   mapping actually applied is printed for transparency, and only then are
   the test columns renamed to their canonical training-side names. If any
   test disease column cannot be matched to a training column this way, the
   notebook stops with a clear error instead of guessing.
5. **Selected-feature availability** in the test symptoms file (checked later
   in Section 8, once the CI stage has decided which symptoms are selected).

In [ ]:
def validate_binary_no_missing(df: pd.DataFrame, label: str):
    errors = []
    for col in df.columns:
        if df[col].isna().any():
            errors.append(f"{label} column '{col}' contains missing values.")
        elif not set(df[col].unique().tolist()) <= {0, 1}:
            errors.append(f"{label} column '{col}' is not strictly binary (0/1). "
                           f"Found values: {sorted(df[col].unique().tolist())}")
    if errors:
        raise ValueError(f"Validation FAILED for {label}:\n" + "\n".join(f"  - {e}" for e in errors))


def validate_row_alignment(df_disease: pd.DataFrame, df_symptoms: pd.DataFrame, label: str):
    """Checks equal row COUNTS only. This is a necessary but not sufficient
    check for row-by-row alignment: it cannot prove row i of df_disease and
    row i of df_symptoms describe the same patient, because neither file
    carries a patient-ID column. That correspondence is assumed to hold
    because both files are assumed to originate from splitting one source
    dataframe's columns, in the original row order, with no reshuffling in
    between. This function does not (and cannot) verify that assumption --
    it only catches the case where it is already violated at the row-count
    level."""
    if df_disease.shape[0] != df_symptoms.shape[0]:
        raise ValueError(f"{label}: disease/symptom row-count mismat ch "
                          f"({df_disease.shape[0]} vs {df_symptoms.shape[0]}).")


def validate_one_hot(df_disease: pd.DataFrame, label: str):
    row_sums = df_disease.sum(axis=1)
    bad_zero = df_disease.index[row_sums == 0].tolist()
    bad_multi = df_disease.index[row_sums > 1].tolist()
    if bad_zero or bad_multi:
        msg = [f"{label}: one-hot validation FAILED."]
        if bad_zero:
            msg.append(f"  - {len(bad_zero)} row(s) with ZERO diseases=1 (rows: {bad_zero[:20]}"
                       f"{' ...' if len(bad_zero) > 20 else ''})")
        if bad_multi:
            msg.append(f"  - {len(bad_multi)} row(s) with MORE THAN ONE disease=1 (rows: {bad_multi[:20]}"
                       f"{' ...' if len(bad_multi) > 20 else ''})")
        raise ValueError("\n".join(msg))
    print(f"{label}: every row has exactly one disease = 1 ({df_disease.shape[0]} rows checked).")


def align_disease_columns(train_disease: pd.DataFrame, test_disease: pd.DataFrame):
    """Map every test disease column onto its training-side counterpart.

    Tries an exact match first. If the column sets differ, falls back to a
    normalized (stripped + lowercased) match used ONLY for comparison; the
    actual applied renaming is printed. Raises if any test column cannot be
    matched to exactly one training column.
    """
    train_cols = list(train_disease.columns)
    test_cols = list(test_disease.columns)

    if set(train_cols) == set(test_cols):
        print("Disease columns match exactly between train and test (no renaming needed).")
        return test_disease, {c: c for c in test_cols}

    norm = lambda c: c.strip().lower()
    train_norm_map = {}
    for c in train_cols:
        train_norm_map.setdefault(norm(c), []).append(c)

    rename_map = {}
    unmatched = []
    ambiguous = []
    for tc in test_cols:
        key = norm(tc)
        matches = train_norm_map.get(key, [])
        if len(matches) == 1:
            rename_map[tc] = matches[0]
        elif len(matches) == 0:
            unmatched.append(tc)
        else:
            ambiguous.append((tc, matches))

    if unmatched or ambiguous:
        msg = ["Disease-class correspondence check FAILED between train and test."]
        if unmatched:
            msg.append(f"  - Test column(s) with no matching training disease (even ignoring "
                       f"case/whitespace): {unmatched}")
        if ambiguous:
            msg.append(f"  - Test column(s) matching more than one training disease: {ambiguous}")
        raise ValueError("\n".join(msg))

    print("Disease columns did not match exactly; applied case/whitespace-normalized alignment:")
    for tc, trc in rename_map.items():
        marker = "  (identical)" if tc == trc else f"  -> renamed to '{trc}'"
        print(f"  '{tc}'{marker}")

    test_disease_aligned = test_disease.rename(columns=rename_map)
    # Reorder to match the training column order for consistency downstream.
    test_disease_aligned = test_disease_aligned[train_cols]
    return test_disease_aligned, rename_map


# --- 1. binary / missing ---
validate_binary_no_missing(train_disease_raw, "training disease dataset")
validate_binary_no_missing(train_symptoms,    "training symptoms dataset")
validate_binary_no_missing(test_disease_raw,  "test disease dataset")
validate_binary_no_missing(test_symptoms,     "test symptoms dataset")

# --- 2. row alignment ---
validate_row_alignment(train_disease_raw, train_symptoms, "training data")
validate_row_alignment(test_disease_raw,  test_symptoms,  "test data")

# --- 3. exactly one disease per patient ---
validate_one_hot(train_disease_raw, "Training disease dataset")
validate_one_hot(test_disease_raw,  "Test disease dataset")

# --- 4. disease-class correspondence (train is the canonical naming) ---
test_disease, disease_rename_map = align_disease_columns(train_disease_raw, test_disease_raw)
train_disease = train_disease_raw

print(f"\nDiseases: {train_disease.shape[1]}  |  Symptoms: {train_symptoms.shape[1]}")
print(f"Training patients: {train_disease.shape[0]}  |  Test patients: {test_disease.shape[0]}")


## 6. CI screening (testing logic retained from `CI_Parallel_multi_disease.ipynb`)

The CI **testing logic** in this section is retained from the reference
notebook: same G-square math, same paired-variable construction, same fixed
Stage-2 search order, same early stopping, same degenerate/invalid handling,
same disease-level parallelization. The *result-handling layer* has been
extended so `process_symptom` / `run_full_screening_for_disease` also return
a detailed, test-by-test record (`all_detailed_ci`) alongside the per-symptom
summary (`all_summaries`) -- for reproducibility, not to change any decision.
It is run once, here, on **`train_disease` / `train_symptoms` only**.

For a dataset this size (41 diseases × 132 symptoms × up to 4,920 rows), this
stage does real work — up to 41 × 132 × 133 ≈ 720,000 individual G-square
tests in the worst case (early stopping keeps the realistic number far lower,
but still substantial). Expect this cell to take a few minutes; the
disease-level parallelism inherited from the reference notebook uses however
many CPU cores are available to shorten that.

The only deliberate change from the reference notebook is display-only:
`build_final_graph` here still computes and saves the exact same PNG/PDF
skeleton per disease, but no longer calls `plt.show()` or prints "Saved: ..."
lines, since inline-rendering 41 figures in this notebook would bury the ML
result the user actually asked for. The decision logic, the retained-symptom
sets, and the saved files themselves are byte-for-byte the same as running
the reference notebook directly.

In [ ]:
def build_paired_variables(df_disease: pd.DataFrame, df_symptoms: pd.DataFrame, disease_col: str) -> pd.DataFrame:
    """Return a DataFrame of paired variables P_j = disease_col AND symptom_j,
    one column per symptom, same column order as df_symptoms."""
    d = df_disease[disease_col]
    paired = pd.DataFrame({
        f"__PAIR__{disease_col}__{symptom}": (d & df_symptoms[symptom]).astype(int)
        for symptom in df_symptoms.columns
    })
    return paired


def build_contingency_table(data: pd.DataFrame, x: str, y: str) -> np.ndarray:
    table = pd.crosstab(data[x], data[y]).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    return table.values.astype(float)


def compute_g_square_from_table(table: np.ndarray):
    """Returns (G2, df, valid, reason). valid=False marks a degenerate table
    (a zero row-sum or column-sum -> one variable does not vary here)."""
    row_sums = table.sum(axis=1)
    col_sums = table.sum(axis=0)
    total = table.sum()

    if total == 0:
        return 0.0, 0, False, "empty stratum: zero observations"
    if (row_sums == 0).any() or (col_sums == 0).any():
        return 0.0, 0, False, "structural zero: one variable is constant within this stratum"

    expected = np.outer(row_sums, col_sums) / total
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(table > 0, table / expected, 1.0)
        terms = np.where(table > 0, table * np.log(ratio), 0.0)
    g2 = float(2.0 * terms.sum())
    df = (table.shape[0] - 1) * (table.shape[1] - 1)

    if not np.isfinite(g2) or df <= 0:
        return 0.0, 0, False, "non-finite G-square statistic or zero degrees of freedom"
    return g2, df, True, "ok"


def _is_binary_and_varying(series: pd.Series) -> bool:
    return len(set(series.unique().tolist())) >= 2


def run_unconditional_test(data: pd.DataFrame, x: str, y: str, alpha: float = ALPHA) -> dict:
    """Stage 1: test X ⟂ Y with an empty conditioning set."""
    result = dict(current_symptom=y, conditioning_variable=None, conditioning_set_size=0,
                  test_stage="Stage1", G_square_statistic=np.nan, degrees_of_freedom=np.nan,
                  p_value=np.nan, alpha=alpha, decision=None, status=None, reason=None)
    if not _is_binary_and_varying(data[x]) or not _is_binary_and_varying(data[y]):
        result.update(status="INVALID", reason=f"'{x}' or '{y}' is constant in the full data",
                       decision="DEPENDENT")
        return result
    table = build_contingency_table(data, x, y)
    g2, df, valid, reason = compute_g_square_from_table(table)
    if not valid:
        result.update(status="INVALID", reason=reason, decision="DEPENDENT")
        return result
    p = float(chi2.sf(g2, df))
    decision = "INDEPENDENT" if p >= alpha else "DEPENDENT"
    result.update(G_square_statistic=g2, degrees_of_freedom=df, p_value=p,
                  status="VALID", reason="ok", decision=decision)
    return result


def run_conditional_test(data: pd.DataFrame, x: str, y: str, z: str, alpha: float = ALPHA) -> dict:
    """Stage 2: test X ⟂ Y | Z with exactly one conditioning variable Z."""
    result = dict(current_symptom=y, conditioning_variable=z, conditioning_set_size=1,
                  test_stage="Stage2", G_square_statistic=np.nan, degrees_of_freedom=np.nan,
                  p_value=np.nan, alpha=alpha, decision=None, status=None, reason=None)
    if not _is_binary_and_varying(data[x]) or not _is_binary_and_varying(data[y]):
        result.update(status="INVALID", reason=f"'{x}' or '{y}' is constant in the full data",
                       decision="DEPENDENT")
        return result

    total_g2, total_df, valid_strata = 0.0, 0, 0
    skipped = []
    for stratum_value in (0, 1):
        stratum = data[data[z] == stratum_value]
        if len(stratum) == 0:
            skipped.append((stratum_value, "empty stratum"))
            continue
        table = build_contingency_table(stratum, x, y)
        g2, df, valid, reason = compute_g_square_from_table(table)
        if not valid:
            skipped.append((stratum_value, reason))
            continue
        total_g2 += g2
        total_df += df
        valid_strata += 1

    if valid_strata == 0 or total_df == 0:
        result.update(status="INVALID",
                       reason=f"no usable stratum conditioning on '{z}': all strata degenerate -> {skipped}",
                       decision="DEPENDENT")
        return result

    p = float(chi2.sf(total_g2, total_df))
    status = "VALID" if not skipped else "DEGENERATE"
    reason = "ok" if not skipped else f"{len(skipped)} stratum/strata skipped: {skipped}"
    decision = "INDEPENDENT" if p >= alpha else "DEPENDENT"
    result.update(G_square_statistic=total_g2, degrees_of_freedom=total_df, p_value=p,
                  status=status, reason=reason, decision=decision)
    return result


def process_symptom(data, disease_col, symptom, paired_vars_order, alpha=ALPHA, verbose=True):
    # ---------------- Stage 1: unconditional CI test ----------------
    r1 = run_unconditional_test(data, disease_col, symptom, alpha=alpha)

    if verbose:
        print(
            f"[{disease_col}] {symptom:22s} | Stage1 | "
            f"p={r1['p_value']:.4g} status={r1['status']:10s} -> {r1['decision']}"
        )

    # Summary information for this disease-symptom pair
    summary = {
        "Disease": disease_col,
        "Symptom": symptom,
        "Initial_p_value": r1["p_value"],
        "Initial_decision": r1["decision"],
        "Stage2_tests_performed": 0,
        "First_removing_conditioning_variable": None,
        "Final_decision": None,
    }

    # Detailed record of every CI test performed
    test_records = []
    test_sequence = 0

    # Save Stage 1 result
    test_sequence += 1
    test_records.append({
        "Disease": disease_col,
        "Symptom": symptom,
        "Test_sequence": test_sequence,
        "Test_stage": r1["test_stage"],
        "Conditioning_variable": r1["conditioning_variable"],
        "Conditioning_variable_type": "NONE",
        "Conditioning_set_size": r1["conditioning_set_size"],
        "G_square_statistic": r1["G_square_statistic"],
        "Degrees_of_freedom": r1["degrees_of_freedom"],
        "P_value": r1["p_value"],
        "Alpha": r1["alpha"],
        "Status": r1["status"],
        "Reason": r1["reason"],
        "Decision": r1["decision"],
    })

    # If Stage 1 finds independence, remove the edge and stop
    if r1["decision"] == "INDEPENDENT":
        summary["Final_decision"] = "REMOVE"
        return summary, test_records

    # ---------------- Stage 2: conditional CI tests ----------------
    for paired_var in paired_vars_order:
        r2 = run_conditional_test(
            data,
            disease_col,
            symptom,
            paired_var,
            alpha=alpha
        )

        summary["Stage2_tests_performed"] += 1

        if verbose:
            short_z = paired_var.split("__")[-1]
            print(
                f"[{disease_col}] {symptom:22s} | Stage2 | "
                f"Z={short_z:22s} p={r2['p_value']:.4g} "
                f"status={r2['status']:10s} -> {r2['decision']}"
            )

        # Save this individual Stage 2 result
        test_sequence += 1
        test_records.append({
            "Disease": disease_col,
            "Symptom": symptom,
            "Test_sequence": test_sequence,
            "Test_stage": r2["test_stage"],
            "Conditioning_variable": (
                paired_var.split("__")[-1]
                if r2["conditioning_variable"] is not None
                else None
            ),
            "Conditioning_variable_type": "PAIRED_AND",
            "Conditioning_set_size": r2["conditioning_set_size"],
            "G_square_statistic": r2["G_square_statistic"],
            "Degrees_of_freedom": r2["degrees_of_freedom"],
            "P_value": r2["p_value"],
            "Alpha": r2["alpha"],
            "Status": r2["status"],
            "Reason": r2["reason"],
            "Decision": r2["decision"],
        })

        # First conditioning variable that establishes independence
        if r2["decision"] == "INDEPENDENT":
            summary["First_removing_conditioning_variable"] = (
                paired_var.split("__")[-1]
            )
            summary["Final_decision"] = "REMOVE"

            return summary, test_records

    # All Stage 2 tests remained dependent
    summary["Final_decision"] = "KEEP"

    return summary, test_records


def run_full_screening_for_disease(
    df_disease,
    df_symptoms,
    disease_col,
    alpha=ALPHA,
    verbose=True
):
    print("=" * 60)
    print(f"Disease: {disease_col}")
    print("=" * 60)

    paired = build_paired_variables(
        df_disease,
        df_symptoms,
        disease_col
    )

    combined = pd.concat(
        [df_disease[[disease_col]], df_symptoms, paired],
        axis=1
    )

    paired_vars_order = list(paired.columns)

    summaries = []
    detailed_test_records = []

    for symptom in df_symptoms.columns:
        summary, test_records = process_symptom(
            combined,
            disease_col,
            symptom,
            paired_vars_order,
            alpha=alpha,
            verbose=verbose
        )

        summaries.append(summary)
        detailed_test_records.extend(test_records)

    summary_df = pd.DataFrame(summaries)
    detailed_ci_df = pd.DataFrame(detailed_test_records)

    retained_symptoms = summary_df.loc[
        summary_df["Final_decision"] == "KEEP",
        "Symptom"
    ].tolist()

    print(f"\n{disease_col}: KEEP {retained_symptoms}\n")

    return summary_df, detailed_ci_df, retained_symptoms


def _wrap_label(name: str, width: int = 10) -> str:
    return textwrap.fill(name.replace("_", " "), width=width)


def build_final_graph(disease_col: str, retained_symptoms: list, save_dir: str):
    """Build and save ONE star-shaped skeleton for one disease: disease_col
    at the center, connected only to its retained (KEEP) symptoms."""
    G = nx.Graph()
    G.add_node(disease_col)
    for symptom in retained_symptoms:
        G.add_edge(disease_col, symptom)

    n = len(retained_symptoms)
    radius = max(2.2, 0.9 + 0.35 * n)

    pos = {disease_col: np.array([0.0, 0.0])}
    for i, symptom in enumerate(retained_symptoms):
        angle = 2 * np.pi * i / max(n, 1)
        pos[symptom] = radius * np.array([np.cos(angle), np.sin(angle)])

    labels = {disease_col: _wrap_label(disease_col, width=12)}
    labels.update({s: _wrap_label(s, width=10) for s in retained_symptoms})

    fig, ax = plt.subplots(figsize=(11, 11))
    nx.draw_networkx_nodes(G, pos, nodelist=[disease_col], node_color="#d62728", node_size=5400, ax=ax)
    if n > 0:
        nx.draw_networkx_nodes(G, pos, nodelist=retained_symptoms, node_color="#1f77b4", node_size=3900, ax=ax)
        nx.draw_networkx_edges(G, pos, width=2.0, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=9, font_color="white", ax=ax)

    ax.set_title(f"skeleton of: {disease_col}", fontsize=16)
    ax.axis("off")
    pad = radius * 0.35 + 0.5
    ax.set_xlim(-radius - pad, radius + pad)
    ax.set_ylim(-radius - pad, radius + pad)
    ax.set_aspect("equal")

    safe_name = disease_col.replace(" ", "_")
    png_path = os.path.join(save_dir, f"{safe_name}_star_graph.png")
    pdf_path = os.path.join(save_dir, f"{safe_name}_star_graph.pdf")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return G


def _screen_and_plot_one_disease(disease_col: str, df_disease: pd.DataFrame,
                                  df_symptoms: pd.DataFrame, alpha: float,
                                  save_dir: str, verbose: bool) -> dict:
    """Run the full, unmodified single-disease pipeline inside a worker
    process and return everything the main process needs to reproduce the
    exact same console output and results as the sequential version."""
    matplotlib.use("Agg", force=True)

    log_buffer = io.StringIO()
    with contextlib.redirect_stdout(log_buffer):
        summary_df, detailed_ci_df, retained_symptoms = run_full_screening_for_disease(
            df_disease, df_symptoms, disease_col, alpha=alpha, verbose=verbose
        )
        build_final_graph(disease_col, retained_symptoms, save_dir)

    safe_name = disease_col.replace(" ", "_")
    return {
        "disease_col": disease_col,
        "summary_df": summary_df,
        "detailed_ci_df": detailed_ci_df,
        "retained_symptoms": retained_symptoms,
        "log_text": log_buffer.getvalue(),
        "png_path": os.path.join(save_dir, f"{safe_name}_star_graph.png"),
        "pdf_path": os.path.join(save_dir, f"{safe_name}_star_graph.pdf"),
    }


def run_all_diseases(df_disease: pd.DataFrame, df_symptoms: pd.DataFrame,
                      alpha: float = ALPHA, save_dir: str = OUTPUT_DIR, verbose: bool = True,
                      max_workers: int = None):
    """Run every disease's screening + graph pipeline in parallel worker
    processes (one disease per task), then replay each disease's console
    output in the ORIGINAL disease-column order -- so the printed log and the
    returned summaries/retained-symptom lists are identical to what the
    sequential version produces. Only wall-clock execution differs."""
    disease_cols = list(df_disease.columns)
    n_cpu = mp.cpu_count() or 1
    if max_workers is None:
        max_workers = max(1, min(len(disease_cols), n_cpu))

    can_fork = "fork" in mp.get_all_start_methods()

    all_summaries, all_detailed_ci, all_retained = {}, {}, {}

    if not can_fork or max_workers <= 1 or len(disease_cols) <= 1:
        print(f"Running {len(disease_cols)} disease(s) sequentially "
              f"({'fork unavailable' if not can_fork else 'nothing to parallelize'}).")
        for disease_col in disease_cols:
            summary_df, detailed_ci_df, retained_symptoms = run_full_screening_for_disease(
                df_disease, df_symptoms, disease_col, alpha=alpha, verbose=verbose
            )
            build_final_graph(disease_col, retained_symptoms, save_dir)
            all_summaries[disease_col] = summary_df
            all_detailed_ci[disease_col] = detailed_ci_df
            all_retained[disease_col] = retained_symptoms
        return all_summaries, all_detailed_ci, all_retained

    print(f"Running {len(disease_cols)} disease(s) in parallel across "
          f"{max_workers} worker process(es) (of {n_cpu} CPU(s) available)...")

    results_by_disease = {}
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as executor:
        futures = {
            executor.submit(_screen_and_plot_one_disease, disease_col, df_disease,
                             df_symptoms, alpha, save_dir, verbose): disease_col
            for disease_col in disease_cols
        }
        for future in as_completed(futures):
            disease_col = futures[future]
            try:
                results_by_disease[disease_col] = future.result()
            except Exception as exc:
                raise RuntimeError(f"Screening failed for disease '{disease_col}'") from exc

    for disease_col in disease_cols:
        res = results_by_disease[disease_col]
        print(res["log_text"], end="")
        all_summaries[disease_col] = res["summary_df"]
        all_detailed_ci[disease_col] = res["detailed_ci_df"]
        all_retained[disease_col] = res["retained_symptoms"]

    return all_summaries, all_detailed_ci, all_retained


# Run CI screening on TRAINING DATA ONLY. Test data is not referenced here.
all_summaries, all_detailed_ci, all_retained = run_all_diseases(
    train_disease, train_symptoms, alpha=ALPHA, save_dir=OUTPUT_DIR, verbose=True
)


In [ ]:
# ================================================================
# DISPLAY-ONLY UPDATE:
# Draw disease-specific star graphs with directed edges:
# Disease -> Symptoms
#
# The existing CI screening results are reused.
# No CI tests or machine learning models are rerun by this cell.
# Existing graph filenames and visual styling are preserved.
# ================================================================

def build_final_graph(disease_col: str, retained_symptoms: list, save_dir: str):
    """Build and save a directed star graph for one disease.

    Every edge points from the disease node towards a retained symptom.
    The existing graph layout, colours, node sizes, labels and filenames
    remain unchanged.
    """

    # CHANGE 1: Use a directed graph instead of an undirected graph.
    G = nx.DiGraph()

    G.add_node(disease_col)

    # The order establishes the direction: disease -> symptom.
    for symptom in retained_symptoms:
        G.add_edge(disease_col, symptom)

    n = len(retained_symptoms)
    radius = max(2.2, 0.9 + 0.35 * n)

    pos = {disease_col: np.array([0.0, 0.0])}

    for i, symptom in enumerate(retained_symptoms):
        angle = 2 * np.pi * i / max(n, 1)
        pos[symptom] = radius * np.array([
            np.cos(angle),
            np.sin(angle)
        ])

    labels = {
        disease_col: _wrap_label(disease_col, width=12)
    }

    labels.update({
        symptom: _wrap_label(symptom, width=10)
        for symptom in retained_symptoms
    })

    fig, ax = plt.subplots(figsize=(11, 11))

    # Preserve the original disease-node appearance.
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=[disease_col],
        node_color="#d62728",
        node_size=5400,
        ax=ax
    )

    if n > 0:

        # Preserve the original symptom-node appearance.
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=retained_symptoms,
            node_color="#1f77b4",
            node_size=3900,
            ax=ax
        )

        # CHANGE 2: Draw directional edges with visible arrowheads.
        # Node sizes are supplied so edges terminate near the node
        # boundaries instead of extending through the circles.
        nx.draw_networkx_edges(
            G,
            pos,
            width=2.0,
            arrows=True,
            arrowstyle="-|>",
            arrowsize=20,
            node_size=[
                5400 if node == disease_col else 3900
                for node in G.nodes()
            ],
            ax=ax
        )

    # Preserve the original labels and formatting.
    nx.draw_networkx_labels(
        G,
        pos,
        labels=labels,
        font_size=9,
        font_color="white",
        ax=ax
    )

    ax.set_title(f"skeleton of: {disease_col}", fontsize=16)
    ax.axis("off")

    pad = radius * 0.35 + 0.5

    ax.set_xlim(-radius - pad, radius + pad)
    ax.set_ylim(-radius - pad, radius + pad)
    ax.set_aspect("equal")

    # Preserve the original output filenames.
    safe_name = disease_col.replace(" ", "_")

    png_path = os.path.join(
        save_dir,
        f"{safe_name}_star_graph.png"
    )

    pdf_path = os.path.join(
        save_dir,
        f"{safe_name}_star_graph.pdf"
    )

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.close(fig)

    return G


# ================================================================
# REGENERATE THE STAR GRAPHS USING EXISTING CI SCREENING RESULTS
# ================================================================

print("Regenerating disease-specific star graphs with directed edges...")

for disease_col, retained_symptoms in all_retained.items():
    build_final_graph(
        disease_col,
        retained_symptoms,
        OUTPUT_DIR
    )

print(f"\nCompleted: {len(all_retained)} disease-specific graphs.")
print(f"Updated graph files are saved in: {OUTPUT_DIR}")
print("Each graph is saved in both PNG and PDF formats.")

## 7. Build multiclass target labels

Each patient's target is the single disease column equal to `1`. This has
already been verified exactly-one-hot for both train and test in Section 5,
so `idxmax` is safe here — it is not doing any correction, just reading off
the already-validated one-hot encoding.

After building the labels, two class-coverage checks run: every training
disease must have at least one training sample (an all-zero disease column
would silently train a class the model can never actually learn), and every
disease that appears in the test set must also appear in the training set
(a disease the model never saw during training cannot be predicted for, and
would otherwise fail confusingly deep inside the evaluation metrics instead
of here, up front, with a clear message).

In [ ]:
def build_multiclass_labels(df_disease: pd.DataFrame) -> pd.Series:
    return df_disease.idxmax(axis=1)


def validate_disease_class_coverage(train_disease: pd.DataFrame, y_train: pd.Series, y_test: pd.Series):
    """Every training disease column must have at least one positive sample,
    and every disease that appears in the test labels must also appear in
    the training labels. Neither condition is guaranteed by the earlier
    one-hot/binary checks, and violating either would make a class
    unlearnable (first case) or unpredictable (second case)."""
    train_sample_counts = train_disease.sum(axis=0)
    diseases_with_no_training_samples = train_sample_counts.index[train_sample_counts == 0].tolist()
    if diseases_with_no_training_samples:
        raise ValueError(
            "Training disease validation FAILED: the following disease column(s) have "
            f"ZERO positive samples in the training data: {diseases_with_no_training_samples}"
        )

    train_classes = set(y_train.unique())
    test_classes = set(y_test.unique())
    test_only_classes = sorted(test_classes - train_classes)
    if test_only_classes:
        raise ValueError(
            "Train/test disease-class validation FAILED: the following disease(s) appear "
            f"in the test set but have NO training samples: {test_only_classes}"
        )

    print(f"Disease-class coverage OK: all {len(train_classes)} training disease(s) have "
          f"at least one training sample; all {len(test_classes)} test disease(s) are "
          f"present in the training data.")


y_train = build_multiclass_labels(train_disease)
y_test = build_multiclass_labels(test_disease)

validate_disease_class_coverage(train_disease, y_train, y_test)

print(f"Training target: {y_train.shape[0]} labels across {y_train.nunique()} disease classes")
print(f"Test target:     {y_test.shape[0]} labels across {y_test.nunique()} disease classes")


## 8. Build CI-informed feature matrix

The CI stage (Section 6) gives a **different** retained symptom set per
disease. A single multiclass model needs **one common feature space**, so we
take the union:



The union is built by walking the symptom columns in their original dataset
order (not set/dict iteration order), so the selected feature list — and
therefore the trained model — is fully deterministic given the same input
data and `ALPHA`.

If CI screening were ever to retain zero symptoms across every disease, the
union would be empty and training would fail deep inside scikit-learn with
a confusing error. That case is checked for explicitly, right after the
union is built, with a clear message instead.

In [ ]:
selected_symptoms = [s for s in train_symptoms.columns
                      if any(s in retained for retained in all_retained.values())]

n_original = train_symptoms.shape[1]
n_selected = len(selected_symptoms)

if n_selected == 0:
    raise ValueError(
        "CI screening retained ZERO symptoms across all diseases -- the unioned "
        "feature set is empty, so a Logistic Regression model cannot be trained. "
        "This does not change the CI algorithm; it only stops execution here, "
        "clearly, instead of failing later inside scikit-learn. Check ALPHA and "
        "the per-disease CI screening output above."
    )

print(f"Original symptoms: {n_original}")
print(f"CI-selected union: {n_selected}")
print(f"Features retained: {n_selected / n_original:.1%}")
print(f"Features removed:  {1 - n_selected / n_original:.1%}")

# The test symptoms file must contain every selected column (already expected,
# since both symptom files share the same column set -- but this is checked
# explicitly rather than assumed).
missing_in_test = [s for s in selected_symptoms if s not in test_symptoms.columns]
if missing_in_test:
    raise ValueError(f"Selected symptom column(s) missing from the test symptoms "
                      f"dataset: {missing_in_test}")

X_train = train_symptoms[selected_symptoms]
X_test = test_symptoms[selected_symptoms]
print(f"\nX_train shape: {X_train.shape}   X_test shape: {X_test.shape}")


## 9. Train multinomial Logistic Regression

**Overfitting prevention:** rather than fitting `LogisticRegression` at a
single fixed regularization strength, `C` is chosen by `GridSearchCV` over
`C_GRID`, using `CV_FOLDS`-fold stratified cross-validation scored on macro
F1 (a good fit for this dataset since every disease class is equally
important and near-equally sized). The search runs entirely on
`X_train`/`y_train` -- it never sees `X_test`/`y_test`, so nothing about the
strict train/test separation established earlier changes. `GridSearchCV`
with `refit=True` (the default) then refits the winning `C` on the full
training set for us; the result is assigned to `model`, so every cell after
this one is unchanged -- it still just uses `model.predict`,
`model.predict_proba`, `model.classes_`, etc., exactly as before.

After fitting, `model.n_iter_` is inspected against `model.max_iter` to
report explicitly whether the solver converged. This doesn't change the
model or its `max_iter` setting -- it only surfaces something scikit-learn
already tracks internally, so a non-converged fit is reported plainly
instead of being treated as a final result.

In [ ]:
# scikit-learn's LogisticRegression no longer exposes a `multi_class`
# parameter (removed as of recent versions) -- the default `lbfgs` solver
# already handles multinomial classification natively, which is the
# current, non-deprecated way to get this behavior.

# Select the L2 regularization strength C by stratified cross-validation on
# TRAINING data only -- this is the overfitting-prevention step. The model
# family and solver are unchanged; only C is searched.
cv_strategy = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid={"C": C_GRID},
    scoring="f1_macro",
    cv=cv_strategy,
    refit=True,
)
grid_search.fit(X_train, y_train)

# GridSearchCV(refit=True) already refit the winning C on the full training
# set -- `model` is that fitted estimator, so every downstream cell that
# uses `model` needs no changes.
model = grid_search.best_estimator_
best_C = grid_search.best_params_["C"]
best_cv_f1_macro = grid_search.best_score_

print(f"Hyperparameter search: C in {C_GRID}, {CV_FOLDS}-fold stratified CV, scoring=macro F1")
print(f"Best C (inverse L2 regularization strength): {best_C}")
print(f"Best cross-validated macro F1: {best_cv_f1_macro:.4f}")
print(f"Model trained on {X_train.shape[0]} patients, {X_train.shape[1]} features, "
      f"{len(model.classes_)} disease classes.")

# Convergence check -- report only, does not alter the fit.
n_iter = model.n_iter_
converged = bool(np.all(n_iter < model.max_iter))
print(f"Solver iterations reported by lbfgs: {n_iter}")
if converged:
    print(f"Solver CONVERGED within max_iter={model.max_iter}.")
else:
    print(f"WARNING: solver did NOT confirm convergence within max_iter={model.max_iter} "
          f"for at least one class. Treat downstream metrics with caution until this is "
          f"resolved (e.g. by increasing max_iter) before relying on this as a final result.")


## 10. Predict disease classes

In [ ]:
y_pred = model.predict(X_test)
print(f"Predicted {len(y_pred)} test patients.")


## 11. Predict disease probabilities

`model.classes_` gives the exact column order of `predict_proba`'s output —
this is confirmed explicitly rather than assumed, since every probability
column downstream depends on it lining up correctly.

In [ ]:
proba = model.predict_proba(X_test)
class_order = model.classes_

assert proba.shape == (X_test.shape[0], len(class_order))
print("model.classes_ (probability column order):")
print(list(class_order))

proba_df = pd.DataFrame(proba, columns=class_order, index=X_test.index)
proba_df.head()


## 12. Generate ranked disease lists

In [ ]:
def rank_diseases_for_row(proba_row: np.ndarray, class_order: np.ndarray):
    order = np.argsort(proba_row)[::-1]
    return [(class_order[i], proba_row[i]) for i in order]


ranked_predictions = [rank_diseases_for_row(proba[i], class_order) for i in range(proba.shape[0])]

print("Example ranking (first test patient):")
for rank, (disease, p) in enumerate(ranked_predictions[0][:5], start=1):
    print(f"  Rank {rank}: {disease:35s} {p:.4f}")


## 13. Evaluate Accuracy / Precision / Recall / F1

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=class_order, average="macro", zero_division=0
)
_, _, weighted_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=class_order, average="weighted", zero_division=0
)

print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall:    {macro_recall:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")

report_dict = classification_report(
    y_test, y_pred, labels=class_order, target_names=class_order,
    output_dict=True, zero_division=0,
)
per_disease_df = (
    pd.DataFrame(report_dict).T
    .loc[list(class_order)]
    .rename(columns={"precision": "Precision", "recall": "Recall",
                      "f1-score": "F1", "support": "Support"})
    [["Precision", "Recall", "F1", "Support"]]
)
per_disease_df.index.name = "Disease"
per_disease_df


## 14. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=class_order)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_order)))
ax.set_yticks(range(len(class_order)))
ax.set_xticklabels(class_order, rotation=90, fontsize=6)
ax.set_yticklabels(class_order, fontsize=6)
ax.set_xlabel("Predicted disease")
ax.set_ylabel("Actual disease")
ax.set_title("Confusion matrix — CI-informed multiclass Logistic Regression")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

confusion_matrix_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
fig.savefig(confusion_matrix_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {confusion_matrix_path}")


## 15. Top-K evaluation

With 41 possible disease classes here, Top-1 (= accuracy), Top-3, and Top-5
are all meaningful. If a dataset has fewer than 5 classes, the largest
meaningful K (at most the number of classes) is used instead.

In [ ]:
n_classes = len(class_order)
k_values = [k for k in (1, 3, 5) if k <= n_classes]

top_k_scores = {}
for k in k_values:
    score = top_k_accuracy_score(y_test, proba, k=k, labels=class_order)
    top_k_scores[k] = score
    print(f"Top-{k} Accuracy: {score:.4f}")


## 16. Final summary

`predict_proba` values are the model's estimated probabilities. Log loss
evaluates the quality of these probabilistic predictions; it does not by
itself establish that the probabilities are well calibrated. This notebook
does not perform a calibration analysis (e.g. reliability curves or
recalibration) -- log loss should be read as a scoring-rule metric on the
predicted probabilities, not as a claim that e.g. "0.82" means a true 82%
real-world likelihood.

In [ ]:
logloss = log_loss(y_test, proba, labels=class_order)

print("FINAL SUMMARY")
print("=" * 60)
print(f"Number of diseases:            {n_classes}")
print(f"Number of original symptoms:   {n_original}")
print(f"Number of CI-selected symptoms:{n_selected}")
print(f"Number of test patients:       {X_test.shape[0]}")
print()
print(f"Accuracy:         {accuracy:.4f}")
print(f"Macro Precision:  {macro_precision:.4f}")
print(f"Macro Recall:     {macro_recall:.4f}")
print(f"Macro F1:         {macro_f1:.4f}")
print(f"Weighted F1:      {weighted_f1:.4f}")
for k, score in top_k_scores.items():
    print(f"Top-{k} Accuracy:    {score:.4f}")
print(f"Log Loss:         {logloss:.4f}")
print()
print("CI-selected symptoms used by the model:")
for s in selected_symptoms:
    print(f"  - {s}")


## 17. Baseline: Logistic Regression on ALL Symptoms (no CI screening)

The model above was trained on the CI-screening-selected symptom subset
(`selected_symptoms`, {n_selected} of {n_original} symptoms -- filled in once this cell
runs). To know whether that feature selection actually helps, we need a baseline: the
same model family, the same hyperparameter-search procedure, the same train/test split
-- trained on **every original symptom column**, with no CI screening applied at all.

`train_symptoms` and `test_symptoms` (loaded in Section 4, before any CI screening) are
reused directly for this -- they already contain the full, unfiltered symptom set.


In [ ]:
# ---------------------------------------------------------------------------
# Align the ALL-symptoms feature set between train and test. The CI-selected feature
# matrix above only ever checked that its (small) selected subset existed in both files;
# here we need every column to match, since nothing is being subset out.
# ---------------------------------------------------------------------------
missing_in_test_full = set(train_symptoms.columns) - set(test_symptoms.columns)
extra_in_test_full = set(test_symptoms.columns) - set(train_symptoms.columns)
if missing_in_test_full or extra_in_test_full:
    raise ValueError(
        "Train/test ALL-symptoms column sets do not match.\n"
        f"  Missing from test: {sorted(missing_in_test_full)}\n"
        f"  Extra in test:     {sorted(extra_in_test_full)}"
    )

X_train_all = train_symptoms[train_symptoms.columns]
X_test_all = test_symptoms[train_symptoms.columns]  # reordered to train's column order

print(f"X_train_all shape: {X_train_all.shape}   X_test_all shape: {X_test_all.shape}")


In [ ]:
# ---------------------------------------------------------------------------
# Train the ALL-symptoms baseline with the SAME hyperparameter-search procedure as the
# CI-informed model above (same C_GRID, same cv_strategy object, same scoring) so the
# comparison isolates the effect of CI-based feature selection rather than differences
# in how each model was tuned.
# ---------------------------------------------------------------------------
grid_search_all = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid={"C": C_GRID},
    scoring="f1_macro",
    cv=cv_strategy,
    refit=True,
)
grid_search_all.fit(X_train_all, y_train)

model_all = grid_search_all.best_estimator_
best_C_all = grid_search_all.best_params_["C"]
best_cv_f1_macro_all = grid_search_all.best_score_

print(f"[All-symptoms baseline] Best C: {best_C_all}  |  Best CV macro F1: {best_cv_f1_macro_all:.4f}")
print(f"[All-symptoms baseline] Trained on {X_train_all.shape[0]} patients, "
      f"{X_train_all.shape[1]} features, {len(model_all.classes_)} disease classes.")

n_iter_all = model_all.n_iter_
converged_all = bool(np.all(n_iter_all < model_all.max_iter))
print(f"[All-symptoms baseline] Solver iterations: {n_iter_all}")
if converged_all:
    print(f"[All-symptoms baseline] Solver CONVERGED within max_iter={model_all.max_iter}.")
else:
    print(f"[All-symptoms baseline] WARNING: solver did NOT confirm convergence within "
          f"max_iter={model_all.max_iter} for at least one class.")


In [ ]:
# ---------------------------------------------------------------------------
# Predict with the all-symptoms baseline
# ---------------------------------------------------------------------------
y_pred_all = model_all.predict(X_test_all)
proba_all = model_all.predict_proba(X_test_all)
class_order_all = model_all.classes_

assert list(class_order_all) == list(class_order), (
    "Class order differs between the CI-informed and all-symptoms models -- this should "
    "not happen since both are trained on the same y_train."
)
print(f"[All-symptoms baseline] Predicted {len(y_pred_all)} test patients.")


## 18. Evaluate the All-Symptoms Baseline

Same metrics as Section 13, computed the same way (`labels=class_order`,
`zero_division=0`), so the two models are directly comparable. `top3_accuracy` for the
CI-informed model is **not** recomputed here -- it's pulled from `top_k_scores[3]`,
already computed in Section 15.


In [ ]:
accuracy_all = accuracy_score(y_test, y_pred_all)

macro_precision_all, macro_recall_all, macro_f1_all, _ = precision_recall_fscore_support(
    y_test, y_pred_all, labels=class_order, average="macro", zero_division=0
)
_, _, weighted_f1_all, _ = precision_recall_fscore_support(
    y_test, y_pred_all, labels=class_order, average="weighted", zero_division=0
)
top3_accuracy_all = (
    top_k_accuracy_score(y_test, proba_all, k=3, labels=class_order)
    if len(class_order) >= 3 else float("nan")
)

# Reuse, don't recompute, the CI-informed model's top-3 accuracy from Section 15.
top3_accuracy_ci = top_k_scores.get(3, float("nan"))

print("[All-symptoms baseline]")
print(f"  Accuracy:        {accuracy_all:.4f}")
print(f"  Macro Precision: {macro_precision_all:.4f}")
print(f"  Macro Recall:    {macro_recall_all:.4f}")
print(f"  Macro F1:        {macro_f1_all:.4f}")
print(f"  Weighted F1:     {weighted_f1_all:.4f}")
print(f"  Top-3 Accuracy:  {top3_accuracy_all:.4f}")


## 19. Comparison: CI-Selected Symptoms vs. All Symptoms

Six metrics, both models, side by side: `accuracy`, `macro_precision`, `macro_recall`,
`macro_f1`, `weighted_f1`, `top3_accuracy`.


In [ ]:
# ---------------------------------------------------------------------------
# model_comparison.csv
# ---------------------------------------------------------------------------
comparison_df = pd.DataFrame([
    {
        "model": "CI-Selected Symptoms",
        "n_features": n_selected,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "top3_accuracy": top3_accuracy_ci,
    },
    {
        "model": "All Symptoms",
        "n_features": n_original,
        "accuracy": accuracy_all,
        "macro_precision": macro_precision_all,
        "macro_recall": macro_recall_all,
        "macro_f1": macro_f1_all,
        "weighted_f1": weighted_f1_all,
        "top3_accuracy": top3_accuracy_all,
    },
])

comparison_path = os.path.join(OUTPUT_DIR, "model_comparison.csv")
comparison_df.to_csv(comparison_path, index=False)
print(f"Saved: {comparison_path}")

all_symptoms_model_path = os.path.join(OUTPUT_DIR, "logistic_regression_model_all_symptoms.joblib")
joblib.dump(model_all, all_symptoms_model_path)
print(f"Saved: {all_symptoms_model_path}")

comparison_df


In [ ]:
# ---------------------------------------------------------------------------
# model_comparison_barplot.png -- grouped bar chart, both models, all six metrics
# ---------------------------------------------------------------------------
metric_cols = [
    ("accuracy", "Accuracy"),
    ("macro_precision", "Macro\nPrecision"),
    ("macro_recall", "Macro\nRecall"),
    ("macro_f1", "Macro\nF1"),
    ("weighted_f1", "Weighted\nF1"),
    ("top3_accuracy", "Top-3\nAccuracy"),
]
bar_labels = [lbl for _, lbl in metric_cols]
ci_values = comparison_df.loc[
    comparison_df["model"] == "CI-Selected Symptoms", [k for k, _ in metric_cols]
].values.flatten().astype(float)
all_values = comparison_df.loc[
    comparison_df["model"] == "All Symptoms", [k for k, _ in metric_cols]
].values.flatten().astype(float)

x = np.arange(len(bar_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars_ci = ax.bar(x - width / 2, ci_values, width,
                  label=f"CI-Selected Symptoms (n={n_selected})", color="#2a9d8f")
bars_all = ax.bar(x + width / 2, all_values, width,
                   label=f"All Symptoms (n={n_original})", color="#e76f51")

for bars in (bars_ci, bars_all):
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.3f}", xy=(b.get_x() + b.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

ax.set_ylabel("Score")
ax.set_ylim(0, 1.08)
ax.set_title("Logistic Regression: CI-Selected Symptoms vs. All Symptoms")
ax.set_xticks(x)
ax.set_xticklabels(bar_labels)
ax.legend(loc="lower right")
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()

barplot_path = os.path.join(OUTPUT_DIR, "model_comparison_barplot.png")
fig.savefig(barplot_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {barplot_path}")


## 20. Overfitting Control: Train vs. Cross-Validation vs. Test

Regularization strength (`C`) is already being tuned by cross-validation in Sections 9
and 17 -- that's the *control* mechanism. What's been missing is the *check*: nothing so
far has looked at how either model performs on the data it was actually trained on, so
there's been no way to see a train/test gap even if one exists.

Four things, for **both** models, using the already-fitted estimators (`model`,
`model_all`) and already-chosen `C` values (`best_C`, `best_C_all`) -- nothing is
retrained with different settings in this section, this is purely diagnostic:

1. **Training-set metrics** -- accuracy / macro F1 / weighted F1 on `X_train`, the same
   data each model was fit on.
2. **`overfitting_gap.csv`** -- Train vs. cross-validated (CV, from the tuning step,
   never touches the test set) vs. Test macro F1 side by side, with the Train-Test gap
   flagged if it exceeds a threshold.
3. **`overfitting_gap_barplot.png`** -- the same three numbers as a grouped bar chart.
4. **Learning curves** -- training-set size vs. score, both a training-score line and a
   cross-validation-score line. A gap between the two lines that stays wide (or widens)
   as training data grows is the classic visual signature of overfitting; both lines
   converging to a low score instead signals underfitting.


In [ ]:
# ---------------------------------------------------------------------------
# Overfitting control, step 1: TRAINING-set performance for both already-fit
# models. Nothing is retrained here -- `model` and `model_all` are the exact
# fitted estimators from Sections 9 and 17.
# ---------------------------------------------------------------------------
y_train_pred = model.predict(X_train)
y_train_pred_all = model_all.predict(X_train_all)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_macro_precision, train_macro_recall, train_macro_f1, _ = precision_recall_fscore_support(
    y_train, y_train_pred, labels=class_order, average="macro", zero_division=0
)
_, _, train_weighted_f1, _ = precision_recall_fscore_support(
    y_train, y_train_pred, labels=class_order, average="weighted", zero_division=0
)

train_accuracy_all = accuracy_score(y_train, y_train_pred_all)
train_macro_precision_all, train_macro_recall_all, train_macro_f1_all, _ = precision_recall_fscore_support(
    y_train, y_train_pred_all, labels=class_order, average="macro", zero_division=0
)
_, _, train_weighted_f1_all, _ = precision_recall_fscore_support(
    y_train, y_train_pred_all, labels=class_order, average="weighted", zero_division=0
)

print("[CI-Selected Symptoms] TRAIN performance:")
print(f"  Accuracy: {train_accuracy:.4f}  Macro F1: {train_macro_f1:.4f}  "
      f"Weighted F1: {train_weighted_f1:.4f}")
print("[All Symptoms] TRAIN performance:")
print(f"  Accuracy: {train_accuracy_all:.4f}  Macro F1: {train_macro_f1_all:.4f}  "
      f"Weighted F1: {train_weighted_f1_all:.4f}")


In [ ]:
# ---------------------------------------------------------------------------
# overfitting_gap.csv -- Train vs. cross-validated vs. Test, both models.
# The CV column is the score GridSearchCV already reported for the WINNING C
# (best_cv_f1_macro / best_cv_f1_macro_all) -- it never touches the test set,
# so it's an independent check that the test-set result isn't a fluke of one
# particular split. Train notably higher than both CV and Test is the classic
# overfitting signature: the model memorized training patterns that neither
# cross-validation folds nor the true test set share.
# ---------------------------------------------------------------------------
OVERFIT_GAP_WARN_THRESHOLD = 0.10  # flag if Train macro-F1 exceeds Test macro-F1 by more than this

overfit_rows = [
    {
        "model": "CI-Selected Symptoms",
        "n_features": n_selected,
        "train_accuracy": train_accuracy,
        "cv_macro_f1": best_cv_f1_macro,
        "train_macro_f1": train_macro_f1,
        "test_macro_f1": macro_f1,
        "train_test_macro_f1_gap": train_macro_f1 - macro_f1,
    },
    {
        "model": "All Symptoms",
        "n_features": n_original,
        "train_accuracy": train_accuracy_all,
        "cv_macro_f1": best_cv_f1_macro_all,
        "train_macro_f1": train_macro_f1_all,
        "test_macro_f1": macro_f1_all,
        "train_test_macro_f1_gap": train_macro_f1_all - macro_f1_all,
    },
]
overfit_df = pd.DataFrame(overfit_rows)

overfit_gap_path = os.path.join(OUTPUT_DIR, "overfitting_gap.csv")
overfit_df.to_csv(overfit_gap_path, index=False)
print(f"Saved: {overfit_gap_path}")

for row in overfit_rows:
    gap = row["train_test_macro_f1_gap"]
    flag = "  <-- POSSIBLE OVERFITTING" if gap > OVERFIT_GAP_WARN_THRESHOLD else ""
    print(f"[{row['model']}] Train macro F1 {row['train_macro_f1']:.4f}  |  "
          f"CV macro F1 {row['cv_macro_f1']:.4f}  |  Test macro F1 {row['test_macro_f1']:.4f}  |  "
          f"Gap (Train-Test) {gap:+.4f}{flag}")

overfit_df


In [ ]:
# ---------------------------------------------------------------------------
# overfitting_gap_barplot.png -- Train / CV / Test macro F1, both models
# ---------------------------------------------------------------------------
groups = ["Train", "CV", "Test"]
ci_scores = [train_macro_f1, best_cv_f1_macro, macro_f1]
all_scores = [train_macro_f1_all, best_cv_f1_macro_all, macro_f1_all]

x = np.arange(len(groups))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
bars_ci = ax.bar(x - width / 2, ci_scores, width,
                  label=f"CI-Selected Symptoms (n={n_selected})", color="#2a9d8f")
bars_all = ax.bar(x + width / 2, all_scores, width,
                   label=f"All Symptoms (n={n_original})", color="#e76f51")

for bars in (bars_ci, bars_all):
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.3f}", xy=(b.get_x() + b.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Macro F1")
ax.set_ylim(0, 1.08)
ax.set_title("Overfitting Check: Train vs. Cross-Validation vs. Test (Macro F1)")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend(loc="lower right")
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()

overfit_barplot_path = os.path.join(OUTPUT_DIR, "overfitting_gap_barplot.png")
fig.savefig(overfit_barplot_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {overfit_barplot_path}")


In [ ]:
# ---------------------------------------------------------------------------
# Learning curves -- training-set size vs. score, both a training-score line
# and a cross-validation-score line, for both models.
#
# Uses the ALREADY-CHOSEN regularization strength (best_C / best_C_all) --
# this diagnoses the model you're actually deploying, it does not re-run the
# hyperparameter search at every training-set size. A separate, smaller CV
# (3-fold, not the 5-fold cv_strategy used for tuning) is used here, since
# the smallest training-set sizes tested may not have enough per-class
# samples for a 5-fold stratified split.
# ---------------------------------------------------------------------------
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, X, y, title, save_path, cv_folds=3):
    cv_lc = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv_lc, scoring="f1_macro",
        train_sizes=np.linspace(0.2, 1.0, 6), random_state=RANDOM_STATE,
    )
    train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
    val_mean, val_std = val_scores.mean(axis=1), val_scores.std(axis=1)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(train_sizes, train_mean, "o-", color="#2a9d8f", label="Training score")
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                     alpha=0.15, color="#2a9d8f")
    ax.plot(train_sizes, val_mean, "o-", color="#e76f51", label="Cross-validation score")
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                     alpha=0.15, color="#e76f51")
    ax.set_xlabel("Training examples")
    ax.set_ylabel("Macro F1")
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right")
    ax.grid(linestyle="--", alpha=0.4)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {save_path}")
    return train_sizes, train_mean, val_mean


lc_ci_path = os.path.join(OUTPUT_DIR, "learning_curve_ci_selected.png")
_ = plot_learning_curve(
    LogisticRegression(C=best_C, max_iter=1000, random_state=RANDOM_STATE),
    X_train, y_train, "Learning Curve -- CI-Selected Symptoms", lc_ci_path,
)

lc_all_path = os.path.join(OUTPUT_DIR, "learning_curve_all_symptoms.png")
_ = plot_learning_curve(
    LogisticRegression(C=best_C_all, max_iter=1000, random_state=RANDOM_STATE),
    X_train_all, y_train, "Learning Curve -- All Symptoms", lc_all_path,
)


## 21. Random Forest and XGBoost

Logistic Regression has now been run both ways (CI-selected symptoms vs. all symptoms).
This section does exactly the same thing for two more model families, so the
CI-vs-all-features question can be answered for each algorithm independently:

| Algorithm | Tuned over | Overfitting control |
|---|---|---|
| Logistic Regression | `C` (inverse L2 strength) | L2 regularization |
| Random Forest | `max_depth`, `min_samples_leaf` | tree-depth / leaf-size limits + bagging |
| XGBoost | `max_depth`, `learning_rate` | tree-depth limit + shrinkage |

Every model uses the **same `cv_strategy` object, the same `f1_macro` scoring, the same
`RANDOM_STATE`, and the same train/test split** as Sections 9 and 17, so the only thing
varying across a comparison is the feature set (and across algorithms, the model family).

**One structural difference for XGBoost:** it requires integer class labels rather than
the disease-name strings used everywhere else, so a `LabelEncoder` is fitted on `y_train`
and predictions are mapped back to disease names immediately. Because the encoder is
fitted on `y_train`, `le.classes_` comes out in the same sorted order as
`model.classes_` / `class_order`, which keeps `proba` column order consistent across all
three algorithms -- that's asserted below rather than assumed.


In [ ]:
# ---------------------------------------------------------------------------
# Hyperparameter grids for the two new model families. Kept deliberately small:
# with ~41 disease classes, XGBoost builds one tree PER CLASS per boosting round,
# so each additional grid point is expensive. Widen these if you have the runtime
# budget -- nothing else needs to change.
# ---------------------------------------------------------------------------
RF_PARAM_GRID = {
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 3],
}
RF_N_ESTIMATORS = 300

XGB_PARAM_GRID = {
    "max_depth": [3, 6],
    "learning_rate": [0.1, 0.3],
}
XGB_N_ESTIMATORS = 200

print("Random Forest grid:", RF_PARAM_GRID, f"(n_estimators={RF_N_ESTIMATORS})")
print("XGBoost grid:      ", XGB_PARAM_GRID, f"(n_estimators={XGB_N_ESTIMATORS})")
print(f"RF fits:  {len(RF_PARAM_GRID['max_depth']) * len(RF_PARAM_GRID['min_samples_leaf'])} "
      f"combos x {CV_FOLDS} folds")
print(f"XGB fits: {len(XGB_PARAM_GRID['max_depth']) * len(XGB_PARAM_GRID['learning_rate'])} "
      f"combos x {CV_FOLDS} folds")


In [ ]:
# ---------------------------------------------------------------------------
# Shared tune -> fit -> evaluate helper.
#
# Every algorithm/feature-set combination goes through this ONE function, which is
# what guarantees the comparison is apples-to-apples: same CV object, same scoring,
# same metric definitions (labels=class_order, zero_division=0), same top-3 rule.
# Writing it once also means a change to the evaluation protocol can't accidentally
# apply to one model but not another.
#
# `label_encoder` is passed only for XGBoost, which needs integer targets.
# ---------------------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

def tune_fit_evaluate(base_estimator, param_grid, X_tr, X_te, y_tr, y_te,
                       algo_name, feature_set_name, label_encoder=None):
    y_tr_fit = label_encoder.transform(y_tr) if label_encoder is not None else y_tr

    gs = GridSearchCV(
        estimator=base_estimator,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv_strategy,
        refit=True,
        n_jobs=-1,
    )
    gs.fit(X_tr, y_tr_fit)
    fitted = gs.best_estimator_

    # --- predictions, mapped back to disease-name space if encoded ---
    raw_pred = fitted.predict(X_te)
    y_pred_ = label_encoder.inverse_transform(raw_pred) if label_encoder is not None else raw_pred
    proba_ = fitted.predict_proba(X_te)

    # Column order of predict_proba follows fitted.classes_; for the encoded case
    # those are integer codes whose names are label_encoder.classes_.
    proba_class_order = (label_encoder.classes_ if label_encoder is not None
                          else fitted.classes_)
    assert list(proba_class_order) == list(class_order), (
        f"[{algo_name} / {feature_set_name}] probability column order does not match "
        f"class_order -- top-3 accuracy would be computed against the wrong columns."
    )

    # --- test metrics (identical definitions to Sections 13 and 18) ---
    acc = accuracy_score(y_te, y_pred_)
    mac_p, mac_r, mac_f1, _ = precision_recall_fscore_support(
        y_te, y_pred_, labels=class_order, average="macro", zero_division=0)
    _, _, wt_f1, _ = precision_recall_fscore_support(
        y_te, y_pred_, labels=class_order, average="weighted", zero_division=0)
    top3 = (top_k_accuracy_score(y_te, proba_, k=3, labels=class_order)
            if len(class_order) >= 3 else float("nan"))

    # --- training-set metrics, for the overfitting gap (Section 20 protocol) ---
    raw_train_pred = fitted.predict(X_tr)
    y_tr_pred_ = (label_encoder.inverse_transform(raw_train_pred)
                   if label_encoder is not None else raw_train_pred)
    train_acc = accuracy_score(y_tr, y_tr_pred_)
    _, _, train_mac_f1, _ = precision_recall_fscore_support(
        y_tr, y_tr_pred_, labels=class_order, average="macro", zero_division=0)

    result = {
        "algorithm": algo_name,
        "feature_set": feature_set_name,
        "n_features": X_tr.shape[1],
        "best_params": gs.best_params_,
        "cv_macro_f1": gs.best_score_,
        "train_accuracy": train_acc,
        "train_macro_f1": train_mac_f1,
        "accuracy": acc,
        "macro_precision": mac_p,
        "macro_recall": mac_r,
        "macro_f1": mac_f1,
        "weighted_f1": wt_f1,
        "top3_accuracy": top3,
        "train_test_macro_f1_gap": train_mac_f1 - mac_f1,
    }

    print(f"[{algo_name} / {feature_set_name}] best={gs.best_params_}  "
          f"CV={gs.best_score_:.4f}  Train={train_mac_f1:.4f}  Test macro F1={mac_f1:.4f}  "
          f"gap={train_mac_f1 - mac_f1:+.4f}")
    return fitted, result

print("tune_fit_evaluate() defined.")


In [ ]:
# ---------------------------------------------------------------------------
# Random Forest -- both feature sets
# ---------------------------------------------------------------------------
rf_model_ci, rf_result_ci = tune_fit_evaluate(
    RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1),
    RF_PARAM_GRID, X_train, X_test, y_train, y_test,
    algo_name="Random Forest", feature_set_name="CI-Selected Symptoms",
)

rf_model_all, rf_result_all = tune_fit_evaluate(
    RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1),
    RF_PARAM_GRID, X_train_all, X_test_all, y_train, y_test,
    algo_name="Random Forest", feature_set_name="All Symptoms",
)


In [ ]:
# ---------------------------------------------------------------------------
# XGBoost -- both feature sets.
# XGBClassifier needs integer targets, so y is label-encoded here and every
# prediction is mapped straight back to disease names inside the helper.
# ---------------------------------------------------------------------------
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_train)
assert list(le.classes_) == list(class_order), (
    "LabelEncoder class order differs from class_order -- probability columns would "
    "be misaligned."
)
print(f"LabelEncoder fitted on {len(le.classes_)} disease classes (order matches class_order).")

def _make_xgb():
    return XGBClassifier(
        n_estimators=XGB_N_ESTIMATORS,
        objective="multi:softprob",
        num_class=len(le.classes_),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="mlogloss",
        tree_method="hist",
    )

xgb_model_ci, xgb_result_ci = tune_fit_evaluate(
    _make_xgb(), XGB_PARAM_GRID, X_train, X_test, y_train, y_test,
    algo_name="XGBoost", feature_set_name="CI-Selected Symptoms", label_encoder=le,
)

xgb_model_all, xgb_result_all = tune_fit_evaluate(
    _make_xgb(), XGB_PARAM_GRID, X_train_all, X_test_all, y_train, y_test,
    algo_name="XGBoost", feature_set_name="All Symptoms", label_encoder=le,
)


## 22. Three-Algorithm Comparison: CI-Selected vs. All Symptoms

One bar chart per algorithm, each comparing the two feature sets across all six metrics
(`accuracy`, `macro_precision`, `macro_recall`, `macro_f1`, `weighted_f1`,
`top3_accuracy`), plus a single combined CSV holding every number.

The Logistic Regression rows are assembled from the metrics already computed in Sections
13, 18 and 20 -- they are **not** recomputed, so the LR bars here will match the Section
19 chart exactly.


In [ ]:
# ---------------------------------------------------------------------------
# Assemble all six model runs (3 algorithms x 2 feature sets) into one table.
# The Logistic Regression rows reuse values already computed earlier -- nothing
# is refit or recomputed for them here.
# ---------------------------------------------------------------------------
lr_result_ci = {
    "algorithm": "Logistic Regression",
    "feature_set": "CI-Selected Symptoms",
    "n_features": n_selected,
    "best_params": {"C": best_C},
    "cv_macro_f1": best_cv_f1_macro,
    "train_accuracy": train_accuracy,
    "train_macro_f1": train_macro_f1,
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "top3_accuracy": top3_accuracy_ci,
    "train_test_macro_f1_gap": train_macro_f1 - macro_f1,
}
lr_result_all = {
    "algorithm": "Logistic Regression",
    "feature_set": "All Symptoms",
    "n_features": n_original,
    "best_params": {"C": best_C_all},
    "cv_macro_f1": best_cv_f1_macro_all,
    "train_accuracy": train_accuracy_all,
    "train_macro_f1": train_macro_f1_all,
    "accuracy": accuracy_all,
    "macro_precision": macro_precision_all,
    "macro_recall": macro_recall_all,
    "macro_f1": macro_f1_all,
    "weighted_f1": weighted_f1_all,
    "top3_accuracy": top3_accuracy_all,
    "train_test_macro_f1_gap": train_macro_f1_all - macro_f1_all,
}

all_results = [
    lr_result_ci, lr_result_all,
    rf_result_ci, rf_result_all,
    xgb_result_ci, xgb_result_all,
]
all_models_df = pd.DataFrame(all_results)

all_models_path = os.path.join(OUTPUT_DIR, "all_models_comparison.csv")
all_models_df.to_csv(all_models_path, index=False)
print(f"Saved: {all_models_path}")

all_models_df


In [ ]:
# ---------------------------------------------------------------------------
# Three bar charts -- one per algorithm, CI-Selected vs. All Symptoms.
# Same metric set, axis range and styling in all three so they can be read
# side by side (or dropped into a report) without rescaling anything.
# ---------------------------------------------------------------------------
BAR_METRICS = [
    ("accuracy", "Accuracy"),
    ("macro_precision", "Macro\nPrecision"),
    ("macro_recall", "Macro\nRecall"),
    ("macro_f1", "Macro\nF1"),
    ("weighted_f1", "Weighted\nF1"),
    ("top3_accuracy", "Top-3\nAccuracy"),
]

def plot_feature_set_comparison(res_ci, res_all, algo_name, save_path):
    labels = [lbl for _, lbl in BAR_METRICS]
    ci_vals = [float(res_ci[k]) for k, _ in BAR_METRICS]
    all_vals = [float(res_all[k]) for k, _ in BAR_METRICS]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    b1 = ax.bar(x - width / 2, ci_vals, width,
                 label=f"CI-Selected Symptoms (n={res_ci['n_features']})", color="#2a9d8f")
    b2 = ax.bar(x + width / 2, all_vals, width,
                 label=f"All Symptoms (n={res_all['n_features']})", color="#e76f51")

    for bars in (b1, b2):
        for b in bars:
            h = b.get_height()
            ax.annotate(f"{h:.3f}", xy=(b.get_x() + b.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=8)

    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.08)
    ax.set_title(f"{algo_name}: CI-Selected Symptoms vs. All Symptoms")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend(loc="lower right")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {save_path}")


plot_feature_set_comparison(
    lr_result_ci, lr_result_all, "Logistic Regression",
    os.path.join(OUTPUT_DIR, "comparison_barplot_logistic_regression.png"))

plot_feature_set_comparison(
    rf_result_ci, rf_result_all, "Random Forest",
    os.path.join(OUTPUT_DIR, "comparison_barplot_random_forest.png"))

plot_feature_set_comparison(
    xgb_result_ci, xgb_result_all, "XGBoost",
    os.path.join(OUTPUT_DIR, "comparison_barplot_xgboost.png"))


In [ ]:
# ---------------------------------------------------------------------------
# Persist the two new model pairs, and print a compact overfitting-gap table
# across all six runs (same Train-vs-Test macro-F1 check as Section 20).
# ---------------------------------------------------------------------------
for obj, fname in [
    (rf_model_ci, "random_forest_model_ci_selected.joblib"),
    (rf_model_all, "random_forest_model_all_symptoms.joblib"),
    (xgb_model_ci, "xgboost_model_ci_selected.joblib"),
    (xgb_model_all, "xgboost_model_all_symptoms.joblib"),
]:
    path = os.path.join(OUTPUT_DIR, fname)
    joblib.dump(obj, path)
    print(f"Saved: {path}")

print("\nOverfitting gap (Train - Test macro F1) across all six runs:")
print(f"{'Algorithm':<22} {'Feature set':<24} {'Train':>7} {'CV':>7} {'Test':>7} {'Gap':>8}")
print("-" * 80)
for r in all_results:
    flag = "  <-- CHECK" if r["train_test_macro_f1_gap"] > OVERFIT_GAP_WARN_THRESHOLD else ""
    print(f"{r['algorithm']:<22} {r['feature_set']:<24} "
          f"{r['train_macro_f1']:>7.4f} {r['cv_macro_f1']:>7.4f} "
          f"{r['macro_f1']:>7.4f} {r['train_test_macro_f1_gap']:>+8.4f}{flag}")


## 23. Save final outputs

The files the user needs for the ML result, per earlier feedback, plus two
additions for research reproducibility:

- `ci_screening_summary.csv` -- one row per disease/symptom pair
  (`Disease`, `Symptom`, `Initial_p_value`, `Initial_decision`,
  `Stage2_tests_performed`, `First_removing_conditioning_variable`,
  `Final_decision`). This is exactly `all_summaries` from Section 6.
- `ci_screening_detailed_results.csv` -- one row per **individual CI test**
  actually performed (every Stage 1 test and every Stage 2 test against
  every paired variable, in the order they ran), with the full G-square
  detail (`Test_stage`, `Conditioning_variable`, `Conditioning_set_size`,
  `G_square_statistic`, `Degrees_of_freedom`, `P_value`, `Alpha`, `Status`,
  `Reason`, `Decision`). This is exactly `all_detailed_ci` from Section 6.

Also saved: `experiment_config.txt` (the exact settings used for this run)
and `logistic_regression_model.joblib` (the fitted model, so predictions can
be reproduced later without retraining). Nothing here is newly computed --
everything was already produced earlier in the notebook; this section only
persists it.

In [ ]:
# --- ci_screening_summary.csv (one row per disease/symptom pair) ---
ci_screening_summary_df = pd.concat(all_summaries.values(), ignore_index=True)
ci_screening_summary_df.to_csv(os.path.join(OUTPUT_DIR, "ci_screening_summary.csv"), index=False)

# --- ci_screening_detailed_results.csv (one row per individual CI test) ---
ci_screening_detailed_df = pd.concat(all_detailed_ci.values(), ignore_index=True)
ci_screening_detailed_df.to_csv(os.path.join(OUTPUT_DIR, "ci_screening_detailed_results.csv"), index=False)

# --- prediction_probabilities.csv ---
pred_df = pd.DataFrame({
    "patient_index": X_test.index,
    "true_disease": y_test.values,
    "predicted_disease": y_pred,
    "probability_of_predicted_disease": proba[np.arange(len(y_pred)), [list(class_order).index(p) for p in y_pred]],
})
proba_cols_df = pd.DataFrame(proba, columns=[f"proba__{c}" for c in class_order], index=X_test.index)
pred_df = pd.concat([pred_df.reset_index(drop=True), proba_cols_df.reset_index(drop=True)], axis=1)
pred_df.to_csv(os.path.join(OUTPUT_DIR, "prediction_probabilities.csv"), index=False)

# --- classification_report.csv ---
per_disease_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report.csv"))

# --- selected_symptoms.txt ---
with open(os.path.join(OUTPUT_DIR, "selected_symptoms.txt"), "w") as f:
    f.write("\n".join(selected_symptoms))

# --- metrics_summary.txt ---
summary_lines = [
    "METRICS SUMMARY",
    "=" * 60,
    f"Number of diseases: {n_classes}",
    f"Number of original symptoms: {n_original}",
    f"Number of CI-selected symptoms: {n_selected}",
    f"Number of test patients: {X_test.shape[0]}",
    "",
    f"Accuracy: {accuracy:.4f}",
    f"Macro Precision: {macro_precision:.4f}",
    f"Macro Recall: {macro_recall:.4f}",
    f"Macro F1: {macro_f1:.4f}",
    f"Weighted F1: {weighted_f1:.4f}",
] + [f"Top-{k} Accuracy: {score:.4f}" for k, score in top_k_scores.items()] + [
    f"Log Loss: {logloss:.4f}",
    "",
    "Solver: lbfgs",
    f"Max iterations: {model.max_iter}",
    f"Iterations used: {[int(x) for x in n_iter]}",
    f"Converged: {'YES' if converged else 'NO'}",
    "",
    f"Regularization strength C (selected via {CV_FOLDS}-fold CV): {best_C}",
    f"Best cross-validated macro F1: {best_cv_f1_macro:.4f}",
]
with open(os.path.join(OUTPUT_DIR, "metrics_summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

# --- experiment_config.txt (exact settings used for this run) ---
# --- experiment_config.txt (exact settings used for this run) ---
config_lines = [
    "EXPERIMENT CONFIGURATION",
    "=" * 60,
    f"ALPHA: {ALPHA}",
    f"RANDOM_STATE: {RANDOM_STATE}",
    f"LogisticRegression solver: {model.solver}",
    f"max_iter: {model.max_iter}",
    f"class_weight: {model.class_weight}",
    f"C (regularization strength, selected via CV): {best_C}",
    f"C_GRID searched: {C_GRID}",
    f"CV_FOLDS: {CV_FOLDS}",
    f"Best cross-validated macro F1: {best_cv_f1_macro:.4f}",
    f"Number of diseases: {n_classes}",
    f"Number of original symptoms: {n_original}",
    f"Number of selected symptoms: {n_selected}",
    f"Number of training patients: {X_train.shape[0]}",
    f"Number of test patients: {X_test.shape[0]}",
    "",
    "=== CI Screening Procedure ===",
    "Stage 1: Unconditional test D ⟂ S",
    "Stage 2: Conditional test D ⟂ S | (D AND S_j)",
    "Stage 2 conditioning set size: 1",
    "Stage 2 stopping rule: Stop at first INDEPENDENT result",
    "INVALID CI test decision: DEPENDENT",
]

with open(os.path.join(OUTPUT_DIR, "experiment_config.txt"), "w") as f:
    f.write("\n".join(config_lines))

# --- logistic_regression_model.joblib (trained model, for future prediction) ---
joblib.dump(model, os.path.join(OUTPUT_DIR, "logistic_regression_model.joblib"))

print(f"Saved to {OUTPUT_DIR}:")
for fname in ["metrics_summary.txt", "classification_report.csv", "confusion_matrix.png",
              "prediction_probabilities.csv", "selected_symptoms.txt", "ci_screening_summary.csv",
              "ci_screening_detailed_results.csv", "experiment_config.txt",
              "logistic_regression_model.joblib", "model_comparison.csv",
              "model_comparison_barplot.png", "logistic_regression_model_all_symptoms.joblib",
              "overfitting_gap.csv", "overfitting_gap_barplot.png",
              "learning_curve_ci_selected.png", "learning_curve_all_symptoms.png",
              "all_models_comparison.csv",
              "comparison_barplot_logistic_regression.png",
              "comparison_barplot_random_forest.png",
              "comparison_barplot_xgboost.png",
              "random_forest_model_ci_selected.joblib",
              "random_forest_model_all_symptoms.joblib",
              "xgboost_model_ci_selected.joblib",
              "xgboost_model_all_symptoms.joblib"]:
    print(f"  - {fname}")


## Example patient prediction

A worked example, using the first test patient, showing the observed
symptoms, the predicted disease, and the full disease ranking.

In [ ]:
example_idx = 0
example_symptoms = X_test.iloc[example_idx]
observed = example_symptoms[example_symptoms == 1].index.tolist()

print("Example patient")
print("-" * 40)
print("Observed symptoms:")
for s in observed:
    print(f"{s} = 1")

print(f"\nActual disease:\n{y_test.iloc[example_idx]}")
print(f"\nPredicted disease:\n{y_pred[example_idx]}")

print("\nDisease ranking:")
for rank, (disease, p) in enumerate(ranked_predictions[example_idx][:5], start=1):
    print(f"{rank}. {disease:35s} {p:.4f}")
